# Creating the document embeddings for Database

In [24]:
import sys
import json
import ast
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import pickle

## Functions to help clean data

In [25]:
"""
extract_legislation_docs.py

Reads a legislation Excel file and produces three parallel lists ready for ChromaDB:
  - docs_to_embed : pdf_file + title + content (string)
  - docs          : content only (string)
  - metadatas     : dict with ChromaDB-ready key/value pairs (no None values)

Usage:
    python extract_legislation_docs.py <path_to_excel.xlsx>

Outputs three JSON files alongside the script:
    docs_to_embed.json
    docs.json
    metadatas.json
"""
# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #

def safe_str(value):
    """Return empty string for None/NaN, otherwise str(value)."""
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


def parse_metadata(raw) -> dict:
    """Parse the Metadata column (JSON string or Python-literal dict)."""
    if not raw or (isinstance(raw, float)):
        return {}
    text = str(raw).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    try:
        return ast.literal_eval(text)
    except Exception:
        return {}


def null_to_empty(value):
    """Convert None/null to empty string for ChromaDB compatibility."""
    if value is None:
        return ""
    s = str(value).strip()
    return "" if s.lower() == "null" else s


# --------------------------------------------------------------------------- #
# Main extraction
# --------------------------------------------------------------------------- #

def extract(excel_path: str):
    df = pd.read_excel(excel_path)

    # Normalise column names: strip whitespace
    df.columns = [c.strip() for c in df.columns]

    # Expected columns (adjust if your file uses different names)
    COL_SECTION  = "Section"
    COL_TITLE    = "Title"
    COL_CONTENT  = "Section Content"
    COL_METADATA = "Metadata"

    docs_to_embed = []
    docs          = []
    metadatas     = []

    for _, row in df.iterrows():
        section  = safe_str(row.get(COL_SECTION,  ""))
        title    = safe_str(row.get(COL_TITLE,    ""))
        content  = safe_str(row.get(COL_CONTENT,  ""))
        meta_raw = row.get(COL_METADATA, "")

        meta = parse_metadata(meta_raw)

        pdf_file = null_to_empty(meta.get("pdf_file"))

        # ------------------------------------------------------------------- #
        # docs_to_embed: pdf_file + title + content
        # ------------------------------------------------------------------- #
        embed_parts = []
        if pdf_file:
            embed_parts.append(f"Source: {pdf_file}")
        if title:
            embed_parts.append(f"Title: {title}")
        if content:
            embed_parts.append(content)
        docs_to_embed.append("\n".join(embed_parts))

        # ------------------------------------------------------------------- #
        # docs: content only
        # ------------------------------------------------------------------- #
        docs.append(content)

        # ------------------------------------------------------------------- #
        # metadatas: ChromaDB-ready dict (no None values)
        # ------------------------------------------------------------------- #
        record = {
            "source_file":  pdf_file,
            "page":         null_to_empty(meta.get("page")),
            "doc_type":     null_to_empty(meta.get("type")),
            "jurisdiction": "New South Wales, Australia",
            "section":      section,
            "part":         null_to_empty(meta.get("part")),
            "division":     null_to_empty(meta.get("division")),
            "subdivision":  null_to_empty(meta.get("subdivision")),
            "schedule":     null_to_empty(meta.get("schedule")),
            "last_updated": null_to_empty(meta.get("last_updated")),
        }
        metadatas.append(record)

    return docs_to_embed, docs, metadatas

In [26]:
docs_to_embed, docs, metadatas = extract("data/acts_regulations.xlsx")

In [30]:
docs[1046]

'(1) No requirement for reasonable refusal for whole transfer or sub-letting The landlord may\nwithhold consent to a transfer or sub-letting relating to the whole tenancy or\nresidential premises whether or not it is reasonable to do so.\n(2) Consent must not be unreasonably withheld for partial transfer or sub-letting The\nlandlord must not unreasonably withhold consent to a transfer of a tenancy or sub-\nletting of premises if the transfer results only in one or more tenants in addition to an\noriginal tenant under the residential tenancy agreement or the partial sub-letting of\nthe residential premises occupied by the tenant.\n(3) Without limiting subsection (2), the landlord is entitled to withhold consent if—\n(a) the number of proposed occupants is more than the number permitted by the\nresidential tenancy agreement or any applicable consent or approval under the\nEnvironmental Planning and Assessment Act 1979, or\n(b) the proposed tenant or sub-tenant is listed on a residential 

In [5]:
print(len(docs_to_embed))

2085


In [6]:
docs_to_embed[0:5]

['Source: anti_discrimination_act_1977\nTitle: Name of Act\nThis Act may be cited as the Anti-Discrimination Act 1977.',
 'Source: anti_discrimination_act_1977\nTitle: Commencement\n(1) This section and section 1 shall commence on the date of assent to this Act.\n(2) Except as provided in subsection (1), this Act shall commence on such day as may be\nappointed by the Governor in respect thereof and as may be notified by proclamation\npublished in the Gazette.',
 'Source: anti_discrimination_act_1977\nTitle: (Repealed)',
 'Source: anti_discrimination_act_1977\nTitle: Definitions\n(1) In this Act, except in so far as the context or subject-matter otherwise indicates or\nrequires—\naccommodation includes residential or business accommodation.\nassociate of a person means—\n(a) any person with whom the person associates, whether socially or in business or\ncommerce, or otherwise, and\n(b) any person who is wholly or mainly dependent on, or a member of the household\nof, the person.\nBoard 

## Embedding the documents

In [9]:
def embed_documents(
    documents: list[str],
    model: SentenceTransformer,
    max_tokens: int = 32768,
    chunk_overlap: int = 100,
) -> list[list[float]]:
    """
    Cleans, chunks (if needed), embeds, and averages embeddings
    for a list of documents using the harrier-oss-v1 model.

    Parameters
    ----------
    documents     : List of raw document strings to embed.
    model         : Loaded SentenceTransformer model instance.
    max_tokens    : Max token limit for the model (default 32,768).
    chunk_overlap : Number of tokens to overlap between chunks
                    to preserve context at boundaries.

    Returns
    -------
    List of embeddings, one per document.
    """

    tokenizer = model.tokenizer

    def clean(text: str) -> str:
        """Basic cleanup before embedding."""
        text = text.strip()
        text = " ".join(text.split())          # collapse whitespace / newlines
        return text

    def chunk_tokens(token_ids: list[int]) -> list[list[int]]:
        """Split a token list into overlapping chunks within model limit."""
        chunks = []
        start = 0
        while start < len(token_ids):
            end = min(start + max_tokens, len(token_ids))
            chunks.append(token_ids[start:end])
            if end == len(token_ids):
                break
            start += max_tokens - chunk_overlap
        return chunks

    def embed_single(text: str) -> np.ndarray:
        """Embed one document, chunking and averaging if over the token limit."""
        token_ids = tokenizer.encode(text, add_special_tokens=False)
        token_count = len(token_ids)

        print(f"  tokens: {token_count}", end="")

        if token_count <= max_tokens:
            # Fits in one pass
            print("  → single embedding")
            return model.encode(text, normalize_embeddings=True)
        else:
            # Split into chunks and average
            chunks = chunk_tokens(token_ids)
            print(f"  → split into {len(chunks)} chunks")

            chunk_texts = [
                tokenizer.decode(chunk, skip_special_tokens=True)
                for chunk in chunks
            ]
            chunk_embeddings = model.encode(
                chunk_texts,
                normalize_embeddings=True,
                batch_size=4,
            )

            # Weighted average by chunk length then re-normalise
            weights = np.array([len(c) for c in chunks], dtype=np.float32)
            weights /= weights.sum()
            averaged = np.average(chunk_embeddings, axis=0, weights=weights)
            averaged /= np.linalg.norm(averaged)   # re-normalise after averaging
            return averaged

    embeddings = []
    for i, doc in enumerate(documents):
        print(f"[{i+1}/{len(documents)}] {doc[:60]}...")
        cleaned = clean(doc)
        embedding = embed_single(cleaned)
        embeddings.append(embedding.tolist())

    print(f"\nDone. {len(embeddings)} embeddings generated.")
    return embeddings

In [10]:
model = SentenceTransformer("local_model")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2947.51it/s]


In [13]:
embeddings = embed_documents(docs_to_embed, model)

[1/2085] Source: anti_discrimination_act_1977
Title: Name of Act
This...
  tokens: 34  → single embedding
[2/2085] Source: anti_discrimination_act_1977
Title: Commencement
(1)...
  tokens: 78  → single embedding
[3/2085] Source: anti_discrimination_act_1977
Title: (Repealed)...
  tokens: 19  → single embedding
[4/2085] Source: anti_discrimination_act_1977
Title: Definitions
(1) ...
  tokens: 1296  → single embedding
[5/2085] Source: anti_discrimination_act_1977
Title: Act done because...
  tokens: 112  → single embedding
[6/2085] Source: anti_discrimination_act_1977
Title: References to ce...
  tokens: 247  → single embedding
[7/2085] Source: anti_discrimination_act_1977
Title: Act binds Crown
...
  tokens: 51  → single embedding
[8/2085] Source: anti_discrimination_act_1977
Title: (Repealed)...
  tokens: 19  → single embedding
[9/2085] Source: anti_discrimination_act_1977
Title: What constitutes...
  tokens: 345  → single embedding
[10/2085] Source: anti_discrimination_act_1977
Title:

In [14]:
print(len(embeddings))

with open(f"data/embeddings_legislation.pkl", "wb") as f:
    pickle.dump(embeddings, f)

2085


# Creating ChromaDB for Legislative documents

In [7]:
print(len(docs))
print(len(metadatas))

2085
2085


In [8]:
# reopen pickle file containing embeddings
with open('data\embeddings_legislation.pkl', 'rb') as file:
    embeddings = pickle.load(file)

print(len(embeddings))

2085


In [10]:
import chromadb

path = 'vdb/tenancy_regulations'

client = chromadb.PersistentClient(path=path)

In [11]:
collection = client.create_collection(
    name="tenancy_documents",
    embedding_function=None
    )

In [12]:
collections = client.list_collections()

In [13]:
collections

[Collection(name=tenancy_documents)]

In [16]:
# creating ids
ids = [str(i) for i in range(1, len(docs) + 1)]

In [17]:
collection.add(
    ids= ids,
    embeddings= embeddings,
    documents= docs,
    metadatas= metadatas,
)

In [18]:
# load model
model = SentenceTransformer("local_model")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2430.84it/s]


In [19]:
# testing with a query
query = ["I have a question about share housing, can I sub-lease a room?"]
query_embeddings = model.encode(query)

collection.query(
    query_embeddings=query_embeddings,
    n_results=3,
)

{'ids': [['1405', '1486', '1046']],
 'embeddings': None,
 'documents': [['',
   '(1) A home owner may, with the written consent of the operator of the community—\n(a) enter into a tenancy agreement for, or otherwise sub-let, the residential site or the\nhome located on it, or\n(b) assign the site agreement.\n(2) The operator must not unreasonably withhold or refuse consent for a tenancy\nagreement or other sub-lease that is proposed to be entered into or granted once\nduring any 3-year period in which the site agreement has effect and is for a term of 12\nmonths or less.\n(3) The operator must not unreasonably withhold or refuse consent to the assignment of\na tenancy agreement.\n(4) Section 133B of the Conveyancing Act 1919 does not prevent the operator from\nwithholding or refusing consent, for any or no reason, for a tenancy agreement or\nother sub-lease if it is for a term exceeding 12 months.\n(5) This section has effect despite the terms of the site agreement and does not prevent